# pulling temperature data 

Primero descargamos el dataset de socrata

Link: https://dev.socrata.com/foundry/www.datos.gov.co/sbwg-7ju4 

Pesa 15GB

In [9]:
import os
import sys
import findspark
import warnings

warnings.filterwarnings("ignore")

# Actualizamos a la ruta de Java 17
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

# Aseguramos que use el Python del entorno de uv
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
try:
    import pyspark
    findspark.init() # findspark buscará automáticamente en .venv
    from pyspark.sql import SparkSession
    
    spark = SparkSession.builder \
        .appName("PruebaSpark") \
        .config("spark.driver.memory", "2g") \
        .getOrCreate()
    
    print("¡Sesión de Spark creada con éxito en Python 3.12!")
    spark.stop()
except Exception as e:
    print(f"Error al iniciar Spark: {e}")

26/05/12 20:33:21 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


¡Sesión de Spark creada con éxito en Python 3.12!


In [14]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# Esquema basado exactamente en los nombres que proporcionaste
schema_soda = StructType([
    StructField("CodigoEstacion", StringType(), True),
    StructField("CodigoSensor", StringType(), True),
    StructField("FechaObservacion", StringType(), True),
    StructField("ValorObservado", DoubleType(), True),
    StructField("NombreEstacion", StringType(), True),
    StructField("Departamento", StringType(), True),
    StructField("Municipio", StringType(), True),
    StructField("ZonaHidrografica", StringType(), True),
    StructField("Latitud", DoubleType(), True),
    StructField("Longitud", DoubleType(), True),
    StructField("DescripcionSensor", StringType(), True),
    StructField("UnidadMedida", StringType(), True)
])

# Lectura del CSV con el esquema correcto
csv_path = "/home/pxtroniwnl/Downloads/temperature-soda.csv"

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .schema(schema_soda) \
    .load(csv_path)

# Verificación de que los datos están en su sitio
print("Verificando alineación de columnas:")
df.show(5)

Verificando alineación de columnas:
+--------------+------------+--------------------+--------------+----------------+---------------+-----------+----------------+-----------+------------+-----------------+------------+
|CodigoEstacion|CodigoSensor|    FechaObservacion|ValorObservado|  NombreEstacion|   Departamento|  Municipio|ZonaHidrografica|    Latitud|    Longitud|DescripcionSensor|UnidadMedida|
+--------------+------------+--------------------+--------------+----------------+---------------+-----------+----------------+-----------+------------+-----------------+------------+
|    0023125501|        0068|2017 Aug 21 05:30...|      23.89136|    PAUNA  - AUT|         BOYACÁ|      PAUNA| MEDIO MAGDALENA|5.657130556|-73.96032778|    Temp Aire 2 m|          °C|
|    0029065000|        0068|2019 Jun 19 06:00...|          30.5|MEDIA LUNA - AUT|      MAGDALENA|    PIVIJAI|  BAJO MAGDALENA|10.51002778|-74.50666667|    Temp Aire 2 m|          °C|
|    0021202180|        0068|2011 Aug 01 01:

In [15]:
# Filtrar por municipio y ver qué estaciones tenemos disponibles
df_local = df.filter(df.Municipio == "CARTAGENA")

# Ver estaciones únicas y su ubicación para seleccionar la mejor
df_local.select("NombreEstacion", "Latitud", "Longitud").distinct().show(truncate=False)

+--------------------------+-------+--------+
|NombreEstacion            |Latitud|Longitud|
+--------------------------+-------+--------+
|APTO RAFAEL NUÑEZ  TX GPRS|10.447 |-75.516 |
+--------------------------+-------+--------+



Ya sabiendo la estacion mas cerca a la laguna lo que procedemos a hacer es filtramos todos los datos que no sirven para nuestra laguna de barcelona de indias

In [18]:
from pyspark.sql import functions as F

# 1. Configuración de sesión y limpieza de fecha
# (Asumiendo que df ya existe con el esquema de 12 columnas)
df_barcelona = df.filter(
    (df.NombreEstacion.contains("RAFAEL NUÑEZ")) & 
    (df.Municipio == "CARTAGENA")
)

df_final = df_barcelona.withColumn(
    "Fecha", 
    F.to_timestamp("FechaObservacion", "yyyy MMM dd hh:mm:ss a")
).filter(F.col("Fecha").isNotNull())

# 2. Exportación a CSV
# Usamos coalesce(1) para que sea un solo archivo
path_temp = "/home/pxtroniwnl/Downloads/temp_folder"

df_final.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .option("sep", ",") \
    .csv(path_temp)

# 3. Truco de terminal para renombrar el archivo a 'datos_temp_barcelona.csv'
import glob

# Buscamos el archivo .csv que generó Spark dentro de la carpeta
generated_file = glob.glob(f"{path_temp}/*.csv")[0]
final_name = "/home/pxtroniwnl/Downloads/datos_temp_barcelona.csv"

os.rename(generated_file, final_name)
print(f"Archivo exportado exitosamente como: {final_name}")

Archivo exportado exitosamente como: /home/pxtroniwnl/Downloads/datos_temp_barcelona.csv
